In [143]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
import matplotlib.pyplot as plt


In [144]:
custo_transporte = pd.read_excel('exerc8.xlsx', sheet_name='custo_transporte')
quantidade_produzida = pd.read_excel('exerc8.xlsx', sheet_name='quantidade_produzida')
demanda_consumidor = pd.read_excel('exerc8.xlsx', sheet_name='demanda_consumidor')


In [145]:
custo_transporte = custo_transporte.drop(columns=['Unidade']).set_index('Produtor')
print(custo_transporte.columns)
custo_transporte.head()



Index(['Mercosul', 'Chile', 'UE', 'Japão', 'Ásia/Pacífico'], dtype='object')


,Mercosul,Chile,UE,Japão,Ásia/Pacífico
Produtor,,,,,
SP I,52,77,145,280,267
SP II,60,85,150,285,272
NE,110,135,115,301,287


In [146]:
quantidade_produzida=quantidade_produzida.set_index('Produtor')
quantidade_produzida.head()

,Produção
Produtor,
SP I,771
SP II,964
NE,193


In [147]:
demanda_consumidor=demanda_consumidor.T
demanda_consumidor = demanda_consumidor[0][2:-1]


## APRENDER A TRANSFORMAR DF IN DICIO

In [148]:
dicio_custo = custo_transporte.T.to_dict()
dicio_quantidade_produzida = quantidade_produzida.T.to_dict()
dicio_demanda_consumidor = demanda_consumidor.to_dict()

In [149]:
lista_produtor = custo_transporte.index
lista_consumidores = custo_transporte.columns

In [150]:
dicio_demanda_consumidor['UE']

1680

In [151]:
model = pyo.ConcreteModel()

model.lista_produtor = pyo.Set(initialize=lista_produtor)
model.lista_consumidores = pyo.Set(initialize=lista_consumidores)

model.x = pyo.Var(model.lista_produtor, model.lista_consumidores, bounds=(0, None), domain=pyo.NonNegativeIntegers)

#objetivo
def objetivo(model):

    return sum(model.x[p,c] * dicio_custo[p][c] for p in model.lista_produtor for c in model.lista_consumidores )
model.objetivo = pyo.Objective(rule=objetivo, sense=pyo.minimize)

#Restrição
def maxima_producao_por_produtora(model,p):
    
    return sum(model.x[p,c] for c in lista_consumidores) == dicio_quantidade_produzida[p]['Produção']
model.maxima_producao_por_produtora = pyo.Constraint(model.lista_produtor, rule=maxima_producao_por_produtora)

def demanda_consumidor(model, c):
   
    return  sum(model.x[p,c] for p in model.lista_produtor) == dicio_demanda_consumidor[c]
model.demanda_consumidor = pyo.Constraint(model.lista_consumidores, rule=demanda_consumidor)



In [152]:
model.pprint()

2 Set Declarations
    lista_consumidores : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    5 : {'Mercosul', 'Chile', 'UE', 'Japão', 'Ásia/Pacífico'}
    lista_produtor : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'SP I', 'SP II', 'NE'}

1 Var Declarations
    x : Size=15, Index=lista_produtor*lista_consumidores
        Key                        : Lower : Value : Upper : Fixed : Stale : Domain
                   ('NE', 'Chile') :     0 :  None :  None : False :  True : NonNegativeIntegers
                   ('NE', 'Japão') :     0 :  None :  None : False :  True : NonNegativeIntegers
                ('NE', 'Mercosul') :     0 :  None :  None : False :  True : NonNegativeIntegers
                      ('NE', 'UE') :     0 :  None :  None : False :  True : NonNegativeIntegers
           ('NE', 'Ásia/Pacífico') :     0 :  None :  None :

In [153]:
opt = SolverFactory('cplex')
resultado = opt.solve(model)

In [154]:
for p in model.lista_produtor:
    for c in model.lista_consumidores:
        print(f'{p,c}: ',pyo.value(model.x[p,c]))

('SP I', 'Mercosul'):  18.0
('SP I', 'Chile'):  7.0
('SP I', 'UE'):  746.0
('SP I', 'Japão'):  0.0
('SP I', 'Ásia/Pacífico'):  0.0
('SP II', 'Mercosul'):  0.0
('SP II', 'Chile'):  0.0
('SP II', 'UE'):  741.0
('SP II', 'Japão'):  159.0
('SP II', 'Ásia/Pacífico'):  64.0
('NE', 'Mercosul'):  0.0
('NE', 'Chile'):  0.0
('NE', 'UE'):  193.0
('NE', 'Japão'):  0.0
('NE', 'Ásia/Pacífico'):  0.0
